#Generative Adverarial Network

This project is building a Generative Adversarial Network (GAN) using PyTorch. The goal is to train a generator and a discriminator network on the MNIST dataset to enable the generator to create new, realistic-looking handwritten digit images. The notebook sets up the model architecture, defines loss functions, and includes a training loop to iteratively improve both networks.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


#config

In [ ]:
DEVICE = 'cuda'

BATCH_SIZE =128

#size of noise vecto
NOISE_VECTOR_DIM = 64
#oprimizer parameters
LR = 0.0002
BETA_1 = 0.5
BETA_2 = 0.99
#Trainig variables
EPOCHS = 20


#Load MNIST Dataset

In [ ]:
from torchvision import datasets, transforms as T

In [ ]:
train_augs = T.Compose([
    T.RandomRotation((-20,+20)),
    T.ToTensor() # (h,w,c) -> (c,h,w)
])

In [ ]:
from PIL.Image import Transform
trainset = datasets.MNIST('MNIST/', download= True, train=True, transform = train_augs)


In [ ]:
image, label = trainset[9000]
plt.imshow(image.squeeze(), cmap='grey')

#Load Dataset Into Batches

In [ ]:
from torch.utils.data import DataLoader
from torchvision.utils import make_grid

In [ ]:
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
print(len(trainloader)) # total number of batches

In [ ]:
for image, label in trainloader:
  print(image.size())
  break

In [ ]:
def show_tensor_images(tensor_img, num_images=16, size=(1,28,28), filename=None):
  unfalt_img = tensor_img.detach().cpu()
  img_grid = make_grid(unfalt_img[:num_images], nrow=4)
  plt.imshow(img_grid.permute(1,2,0).squeeze())
  if filename:
    plt.savefig(filename)
  plt.show()

In [ ]:
dataiter = iter(trainloader)
images, _ = next(dataiter)
show_tensor_images(images, num_images=16)

#Creater Disciminator Network

In [ ]:
from torch import nn
from torchsummary import summary

In [ ]:
'''
input : (bs, 1, 28, 28)
      |
      v
Conv2d(in_ch = 1, out_channel = 16, kernel_size=(3,3), stride =2)
BatchNorm2d()
LeakyReLU()
      |
      v
Conv2d(in_ch = 16, out_channel =32, kernel_size=(5,5), stride =2)
BatchNorm2d()
LeakyReLU()
      |
      v
Conv2d(in_ch = 32, out_channel =64, kernel_size=(5,5), stride =2)
BatchNorm2d()
LeakyReLU()
      |
      v
Flatten()
Linear(in_ch =64, out_features= 1)

''';

In [ ]:
def get_disc_block(in_channels, out_channels, kernel_size, stride):
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size, stride),
      nn.BatchNorm2d(out_channels),
      nn.LeakyReLU(0.2)
  )

In [ ]:
class Discriminator(nn.Module):
  def __init__(self):
    super(Discriminator, self).__init__()
    self.block_1 = get_disc_block(1, 16,(3,3), 2)
    self.block_2 = get_disc_block(16, 32,(5,5), 2)
    self.block_3 = get_disc_block(32, 64,(5,5), 2)
    self.flatten = nn.Flatten()
    self.linear = nn.Linear(in_features=64, out_features=1)

  def forward(self, images):
    x1 = self.block_1(images)
    x2 = self.block_2(x1)
    x3 = self.block_3(x2)
    x4 = self.flatten(x3)
    x5 = self.linear(x4)
    return x5

In [ ]:
D = Discriminator().to(DEVICE)
summary(D, (1, 28, 28), device=DEVICE)

#Create Generator

In [ ]:
'''
Generator

noise_dim = 64
input : (bs,noise_dim)
      |
      | Reshape
      v
input : (bs,c,h,w) -> (bs, noise_dim, 1, 1)
      |
      v
ConvTranspose2d(in_channels = noise_dim, out_channels = 256, kernel_size=(3,3), stride=2)
BatchNorm2d()
ReLU()
      |
      v
ConvTranspose2d(in_channels = 256, out_channels = 128, kernel_size=(4,4), stride=1)
BatchNorm2d()
ReLU()
      |
      v
ConvTranspose2d(in_channels = 128, out_channels =64, kernel_size=(3,3), stride=2)
BatchNorm2d()
ReLU()
      |
      v
ConvTranspose2d(in_channels = 64, out_channels =1, kernel_size=(4,4), stride=2)
Tanh()
''';

In [ ]:
def get_gen_block(in_channels, out_channels, kernel_size, stride, final_block=False):
  if final_block:
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride),
        nn.Tanh()
    )

  return nn.Sequential(
      nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride),
      nn.BatchNorm2d(out_channels),
      nn.ReLU()
  )

In [ ]:
class Generator(nn.Module):
  def __init__(self, noise_dim=NOISE_VECTOR_DIM):
    super(Generator, self).__init__()
    self.noise_vec_dim = NOISE_VECTOR_DIM
    self.block1 = get_gen_block(noise_dim, 256, (3,3), 2)
    self.block2 = get_gen_block(256, 128, (4,4),1)
    self.block3 = get_gen_block(128, 64, (3,3),2)
    self.block4 = get_gen_block(64, 1, (4,4),2, final_block=True)

  def forward(self, rnd_noise_vec):
    #first convert noise vector from ( bs, noise_dim) -> (bs,noise_dim,1,1)
    x = rnd_noise_vec.view(-1, self.noise_vec_dim, 1, 1)

    x1 = self.block1(x)
    x2 = self.block2(x1)
    x3 = self.block3(x2)
    x4 = self.block4(x3)
    return x4


In [ ]:
G = Generator(NOISE_VECTOR_DIM).to(DEVICE)
summary(G, input_size= ( 1, NOISE_VECTOR_DIM))

In [ ]:
# Replace random initialized weights to normal weights
def weights_init(m):
  if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
    nn.init.normal_(m.weight, 0.0, 0.02)
  if isinstance(m, nn.BatchNorm2d):
    nn.init.normal_(m.weight, 0.0, 0.02)
    nn.init.constant_(m.bias, 0)

In [ ]:
D = D.apply(weights_init)
G = G.apply(weights_init)

#Create Loss Function and Load Optimizer

In [ ]:
#define real loss  for discriminator prediction

def real_loss(disc_pred):
  criterion = nn.BCEWithLogitsLoss()
  ground_truth = torch.ones_like(disc_pred)
  loss = criterion(disc_pred, ground_truth)
  return loss

def fake_loss(disc_pred):
  criterion = nn.BCEWithLogitsLoss()
  ground_truth = torch.zeros_like(disc_pred)
  loss = criterion(disc_pred, ground_truth)
  return loss


In [ ]:
D_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA_1, BETA_2))
G_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA_1, BETA_2))

#Training Loop

In [ ]:
for epoch in range(EPOCHS):
  total_d_loss = 0.0
  total_g_loss = 0.0

  for real_img, _  in tqdm(trainloader):
    real_img = real_img.to(DEVICE)
    noise = torch.randn(BATCH_SIZE, NOISE_VECTOR_DIM, device=DEVICE)

    # First discriminator network
    D_opt.zero_grad()
    fake_img = G(noise)
    d_pred = D(fake_img)
    d_fake_loss = fake_loss(d_pred)
    # now we pass the real image to discriminator
    d_pred = D(real_img)
    d_real_loss = real_loss(d_pred)
    d_loss = d_fake_loss + d_real_loss /2
    total_d_loss += d_loss.item()
    d_loss.backward()
    D_opt.step()

    #Now the Generator Network
    G_opt.zero_grad()
    noise = torch.randn(BATCH_SIZE, NOISE_VECTOR_DIM, device=DEVICE)
    fake_img = G(noise)
    #pass fake image to discriminator
    d_pred = D(fake_img)
    g_loss = real_loss(d_pred)
    total_g_loss += g_loss.item()
    g_loss.backward()
    G_opt.step()

  avg_d_loss = total_d_loss / len(trainloader)
  avg_g_loss = total_g_loss / len(trainloader)

  print(f'epoch: {epoch} | D_loss : {avg_d_loss} | G_loss: {avg_g_loss}')

  # Save the generated images at the end of each epoch
  show_tensor_images(fake_img, filename=f'generated_images_epoch_{epoch:03d}.png')

Created by Carlos Sarasty